# 22 - Prepare Replay Dataset

## Purpose

Prepare a dedicated replay dataset for Kafka streaming.

This notebook does **not** generate timestamps.

The replay producer is responsible for assigning realistic event timestamps during streaming.

### Input

data/processed/creditcard_feature_engineered.csv

### Output

data/replay/creditcard_replay.csv

### Operations

- Validate dataset
- Shuffle transactions
- Generate Replay ID
- Generate Replay Batch
- Export replay dataset

# ============================================
# Section 1 - Imports
# ============================================

In [1]:


import os
import logging

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


# ============================================
# Section 2 - Configuration
# ============================================

In [2]:
INPUT_DATASET = "../data/processed/creditcard_feature_engineered.csv"

OUTPUT_DIRECTORY = "../data/replay"

OUTPUT_DATASET = os.path.join(
    OUTPUT_DIRECTORY,
    "creditcard_replay.csv"
)

LOG_DIRECTORY = "../logs"

LOG_FILE = os.path.join(
    LOG_DIRECTORY,
    "prepare_replay_dataset.log"
)

RANDOM_STATE = 42

ROWS_PER_BATCH = 1000

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)
os.makedirs(LOG_DIRECTORY, exist_ok=True)

print("Configuration loaded.")

Configuration loaded.


# ============================================
# Section 3 - Logging
# ============================================

In [3]:
logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    filemode="a"

)

logging.info("="*80)
logging.info("Replay Dataset Preparation Started")
logging.info("="*80)

print("Logging initialized.")

Logging initialized.


# ============================================
# Section 4 - Verify Input Dataset
# ============================================

In [4]:
if not os.path.exists(INPUT_DATASET):

    raise FileNotFoundError(
        f"Dataset not found:\n{INPUT_DATASET}"
    )

print("Dataset Found.")

logging.info("Dataset verified.")

Dataset Found.


# ============================================
# Section 5 - Load Dataset
# ============================================

In [5]:

df = pd.read_csv(INPUT_DATASET)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

logging.info(f"Rows Loaded : {len(df)}")
logging.info(f"Columns : {len(df.columns)}")

Rows    : 283726
Columns : 34


# ============================================
# Section 6 - Validate Dataset
# ============================================

In [6]:
print("="*60)
print("Dataset Validation")
print("="*60)

print()

print("Missing Values")

print(df.isnull().sum().sum())

print()

print("Duplicate Rows")

print(df.duplicated().sum())

logging.info("Dataset validation completed.")

Dataset Validation

Missing Values
0

Duplicate Rows
0


# ============================================
# Section 7 - Shuffle Dataset
# ============================================

In [7]:


replay_df = (

    df.sample(

        frac=1,

        random_state=RANDOM_STATE

    )

    .reset_index(drop=True)

)

logging.info("Dataset shuffled.")

# ============================================
# Section 8 - Replay Metadata
# ============================================

In [8]:


replay_df.insert(

    0,

    "Replay_ID",

    np.arange(1, len(replay_df)+1)

)

replay_df.insert(

    1,

    "Replay_Batch",

    ((replay_df["Replay_ID"]-1)//ROWS_PER_BATCH)+1

)

logging.info("Replay metadata created.")

# ============================================
# Section 9 - Save Replay Dataset
# ============================================

In [9]:


replay_df.to_csv(

    OUTPUT_DATASET,

    index=False

)

logging.info("Replay dataset exported.")

# ============================================
# Section 10 - Validate Output
# ============================================

In [10]:


saved_df = pd.read_csv(OUTPUT_DATASET)

assert len(saved_df) == len(replay_df)

assert "Replay_ID" in saved_df.columns

assert "Replay_Batch" in saved_df.columns

print()

print("Replay Dataset Validation Passed")

print()

print(saved_df.head())

logging.info("Replay dataset validated.")


Replay Dataset Validation Passed

   Replay_ID  Replay_Batch      Time        V1        V2        V3        V4  \
0          1             1 -0.539502  1.054379 -0.764756  0.160168  0.665587   
1          2             1 -0.295741 -4.805134  4.351191 -0.916135 -0.900752   
2          3             1 -1.129486 -1.549833 -0.261143  1.556289 -2.037817   
3          4             1 -1.982796  0.216344  0.663182  1.303520  0.169219   
4          5             1  1.391272 -0.273365  0.825649  0.555674  0.384915   

         V5        V6        V7  ...       V24       V25       V26       V27  \
0  1.113466  4.771656 -1.536609  ...  1.047201  0.698595 -0.217939  0.101970   
1 -0.870777 -0.782946  0.328104  ... -0.021274  0.552660  0.008318  0.431835   
2 -0.366315 -1.334314  0.165406  ...  0.424100  0.081845  1.103559 -0.502271   
3 -0.404266 -0.517288 -0.036938  ...  0.525962  0.750877  0.141543  0.045107   
4  1.343842  1.984159  0.321808  ... -0.896146  0.503553  0.555129  0.129099   

   

# ============================================
# Section 11 - Summary
# ============================================

In [11]:


print("="*70)

print("Replay Dataset Created Successfully")

print("="*70)

print(f"Rows                : {len(replay_df)}")

print(f"Columns             : {len(replay_df.columns)}")

print(f"Replay Batches      : {replay_df['Replay_Batch'].nunique()}")

print(f"Output File         : {OUTPUT_DATASET}")

print()

print("Timestamp Generation : Producer")

print("Streaming           : Kafka Producer")

print("="*70)

logging.info("Replay dataset preparation completed.")
logging.info("="*80)

Replay Dataset Created Successfully
Rows                : 283726
Columns             : 36
Replay Batches      : 284
Output File         : ../data/replay/creditcard_replay.csv

Timestamp Generation : Producer
Streaming           : Kafka Producer
